# One Example, Last Two Layers Only: custom 3CNN vs. ResNet18

This notebook is intentionally simple.

It shows only one concrete validation sample and only the last two layers of the two models:
- the penultimate layer (`pre_logits`)
- the final output layer (`Dense(... => 2)`)

For the final layer, the notebook prints the concrete values for both classes:
- input activation from the penultimate layer
- weight of the output neuron
- product `input * weight`
- bias
- summed logit
- softmax probability as the final output after activation

The same real-data-only training pipeline from `semi_supervised_learing.ipynb` is still used in the background.


In [1]:
import Pkg

model_test_dir = if isfile(joinpath(pwd(), "Project.toml")) && isfile(joinpath(pwd(), "semi_supervised_learing.ipynb"))
    pwd()
else
    candidate = joinpath(pwd(), "notebooks", "model_test")
    @assert isfile(joinpath(candidate, "Project.toml")) "Could not locate notebooks/model_test from current working directory."
    candidate
end

cd(model_test_dir)
Pkg.activate(model_test_dir)

include(joinpath(pwd(), "test_outputlayer_helpers.jl"))
using .TestOutputLayerHelpers
using PrettyTables

table_kwargs = TestOutputLayerHelpers.TABLE_KWARGS

println("Model-test directory: ", pwd())
println("CUDA available: ", TestOutputLayerHelpers.USE_CUDA)


  Activating project at `~/Dokumente/BA2/notebooks/model_test`


Model-test directory: /home/benjamin/Dokumente/BA2/notebooks/model_test
CUDA available: true


In [2]:
ctx = run_outputlayer_analysis(
    analysis_fold = 1,
    data_split_seed = 20260308,
    fold_seed = 20260220,
    cnn3_epochs = 6,
    resnet_epochs = 4,
    cnn3_lr = 1f-3,
    resnet_lr = 1f-4,
    cnn3_batchsize = 32,
    resnet_batchsize = 16,
)

cnn_run = ctx.analysis_runs["cnn_3conv"]
resnet_run = ctx.analysis_runs["resnet18_pretrained_1ch"]

selected_val_row = first(ctx.interesting_val_rows)

selected_sample_df = cnn_run.predictions[[selected_val_row], [
    :val_row,
    :sample_id,
    :group_id,
    :image_id,
    :sort_var,
    :variant,
    :true_name,
]]

println("\nSelected validation sample:")
pretty_table(selected_sample_df; table_kwargs...)

nothing


[ Info: Running output-layer analysis for cnn_3conv
[ Info: cnn_3conv | epoch 1/6 | train_loss=0.623
[ Info: cnn_3conv | epoch 2/6 | train_loss=0.6043
[ Info: cnn_3conv | epoch 3/6 | train_loss=0.65013
[ Info: cnn_3conv | epoch 4/6 | train_loss=0.61373
[ Info: cnn_3conv | epoch 5/6 | train_loss=0.60361
[ Info: cnn_3conv | epoch 6/6 | train_loss=0.60351
[ Info: Running output-layer analysis for resnet18_pretrained_1ch
[ Info: resnet18_pretrained_1ch | epoch 1/4 | train_loss=0.81136
[ Info: resnet18_pretrained_1ch | epoch 2/4 | train_loss=0.19981
[ Info: resnet18_pretrained_1ch | epoch 3/4 | train_loss=0.14463
[ Info: resnet18_pretrained_1ch | epoch 4/4 | train_loss=0.09404



Selected validation sample:
┌─────────┬───────────┬──────────┬───────────────────────────────────────┬──────────┬────────────┬───────────┐
│ val_row │ sample_id │ group_id │                              image_id │ sort_var │    variant │ true_name │
│   Int64 │     Int64 │    Int64 │                                String │   String │     String │    String │
├─────────┼───────────┼──────────┼───────────────────────────────────────┼──────────┼────────────┼───────────┤
│       1 │         4 │      101 │ erp_additional_001_ch002_duration.png │ duration │ mod4_part3 │ erp_class │
└─────────┴───────────┴──────────┴───────────────────────────────────────┴──────────┴────────────┴───────────┘


## custom 3CNN

Interpretation for this model:
- penultimate layer: a vector with `64` neuron activations
- final layer: `Dense(64 => 2)` with one output neuron for `no_class` and one for `erp_class`
- there is no activation inside the final dense layer itself
- the probabilities are obtained afterwards with `softmax`

So in the tables below:
- `activation_value` is the concrete value in the penultimate layer
- `contribution` is exactly `activation_value * weight`
- `manual_logit` is the sum of all contributions plus bias
- `softmax_probability` is the output after the final activation function


In [3]:
cnn_penultimate_df = instance_penultimate_layer_df(
    cnn_run,
    selected_val_row;
    sort_by = :neuron_idx,
)

cnn_final_summary_df = instance_neuron_summary_df(cnn_run, selected_val_row)

cnn_no_class_df = instance_neuron_breakdown_df(
    cnn_run,
    selected_val_row,
    1;
    sort_by = :feature_idx,
)

cnn_erp_class_df = instance_neuron_breakdown_df(
    cnn_run,
    selected_val_row,
    2;
    sort_by = :feature_idx,
)

println("\nPenultimate layer values: custom 3CNN")
pretty_table(cnn_penultimate_df; table_kwargs...)

println("\nFinal output neurons summary: custom 3CNN")
pretty_table(cnn_final_summary_df; table_kwargs...)

println("\nFinal output neuron = no_class | every input * weight value before the sum")
pretty_table(cnn_no_class_df; table_kwargs...)

println("\nFinal output neuron = erp_class | every input * weight value before the sum")
pretty_table(cnn_erp_class_df; table_kwargs...)

nothing



Penultimate layer values: custom 3CNN
┌────────────┬──────────────────┬──────────────────────┬─────────────────┐
│ neuron_idx │ activation_value │ abs_activation_value │ activation_sign │
│      Int64 │          Float64 │              Float64 │          String │
├────────────┼──────────────────┼──────────────────────┼─────────────────┤
│          1 │              0.0 │                  0.0 │        positive │
│          2 │         0.138842 │             0.138842 │        positive │
│          3 │         0.209739 │             0.209739 │        positive │
│          4 │              0.0 │                  0.0 │        positive │
│          5 │              0.0 │                  0.0 │        positive │
│          6 │         0.224011 │             0.224011 │        positive │
│          7 │       5.39651e-5 │           5.39651e-5 │        positive │
│          8 │         0.164976 │             0.164976 │        positive │
│          9 │              0.0 │                  0.0 │     

## ResNet18 (1 channel)

Interpretation for this model:
- penultimate layer: a vector with `512` neuron activations
- final layer: `Dense(512 => 2)` with one output neuron for `no_class` and one for `erp_class`
- again, the final dense layer produces raw logits and `softmax` turns them into probabilities

Because the penultimate and final vectors are much larger here, the notebook prints the strongest values directly and keeps the full `512` entries in variables.


In [4]:
resnet_penultimate_top_df = instance_penultimate_layer_df(
    resnet_run,
    selected_val_row;
    sort_by = :abs_activation_value,
    top_k = 64,
)

resnet_penultimate_full_df = instance_penultimate_layer_df(
    resnet_run,
    selected_val_row;
    sort_by = :neuron_idx,
)

resnet_final_summary_df = instance_neuron_summary_df(resnet_run, selected_val_row)

resnet_no_class_top_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    1;
    sort_by = :abs_contribution,
    top_k = 32,
)

resnet_erp_class_top_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    2;
    sort_by = :abs_contribution,
    top_k = 32,
)

resnet_no_class_full_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    1;
    sort_by = :feature_idx,
)

resnet_erp_class_full_df = instance_neuron_breakdown_df(
    resnet_run,
    selected_val_row,
    2;
    sort_by = :feature_idx,
)

println("\nPenultimate layer values: ResNet18 | top 64 activations by absolute value")
pretty_table(resnet_penultimate_top_df; table_kwargs...)

println("\nFinal output neurons summary: ResNet18")
pretty_table(resnet_final_summary_df; table_kwargs...)

println("\nFinal output neuron = no_class | top 32 input * weight values before the sum")
pretty_table(resnet_no_class_top_df; table_kwargs...)

println("\nFinal output neuron = erp_class | top 32 input * weight values before the sum")
pretty_table(resnet_erp_class_top_df; table_kwargs...)

println("\n`resnet_penultimate_full_df`, `resnet_no_class_full_df`, and `resnet_erp_class_full_df` contain all 512 values in original order.")
nothing



Penultimate layer values: ResNet18 | top 64 activations by absolute value
┌────────────┬──────────────────┬──────────────────────┬─────────────────┐
│ neuron_idx │ activation_value │ abs_activation_value │ activation_sign │
│      Int64 │          Float64 │              Float64 │          String │
├────────────┼──────────────────┼──────────────────────┼─────────────────┤
│        499 │          5.75042 │              5.75042 │        positive │
│        186 │          5.66359 │              5.66359 │        positive │
│        381 │           5.6067 │               5.6067 │        positive │
│        281 │          5.25777 │              5.25777 │        positive │
│        371 │          5.21931 │              5.21931 │        positive │
│         32 │          4.94622 │              4.94622 │        positive │
│        311 │          4.91362 │              4.91362 │        positive │
│        317 │          4.52103 │              4.52103 │        positive │
│        250 │          4